# Data Collection and Annotation

In [16]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
import re

# Language mapping
languages = {
    'Arabic (Egyptian)': '1037494',
    'Arabic (Levantine)': '1037493',
    'Arabic (Maghrebi)': '1037522'
}

base_url = 'https://lyricstranslate.com/en/songs/{lang_id}/none/none/0/none/0'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def get_songs_for_language(lang_name, lang_id):
    songs = []
    page = 1
    while True:
        # print(f"Fetching page {page} for language ID {lang_id}...")
        url = base_url.format(lang_id=lang_id) + (f'?page={page}' if page > 1 else '')
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            break
        soup = BeautifulSoup(response.text, 'html.parser')
        table = soup.find('div', class_='t-s-r')
        if not table:
            break
        rows = table.find_all('div', class_='d-tr')[1:]  # Skip header
        if not rows:
            break
        for row in rows:
            cells = row.find_all('div')
            # print(cells)
            # exit()
            if "," in cells[4].text.strip():
                continue  # Skip songs with multiple languages
            artist_a = cells[1].find('div', class_='att')
            song_a = cells[2].find('a')
            if artist_a and song_a:
                artist = artist_a.text.strip()
                song = song_a.text.strip()
                song_url = 'https://lyricstranslate.com' + song_a['href']
                artist = str(artist)
                artist = artist.replace('<div class="att">', "").replace("</div>", "")
                songs.append((artist, song, song_url))
        page += 1
        if page == 6:  # Limit to first 6 pages
            break
    return songs

def extract_lyrics_and_check_language(song_url, target_lang):
    response = requests.get(song_url, headers=headers)
    if response.status_code != 200:
        return None, None
    soup = BeautifulSoup(response.text, 'html.parser')
    song_node_text = soup.find('div', class_='song-node-text')
    song_body = song_node_text.find('div', id='song-body') 
    lyrics = song_body.get_text(separator=" ", strip=True)
    return lyrics

def main():
    all_data = []
    unique_id = 0
    lang_count = 0
    for lang_name, lang_id in languages.items():
        print(f"Processing {lang_name}...")
        songs = get_songs_for_language(lang_name, lang_id)
        for artist, song, url in songs:
            song_lang = ""
            lyrics = extract_lyrics_and_check_language(url, lang_name)
            if lang_id == "1037494":
                song_lang = "Egyptian"
            elif lang_id == "1037493":
                song_lang = "Levantine"
            elif lang_id == "1037522":
                song_lang = "Maghrebi"
            if lyrics:
                all_data.append({
                    'ID': f"{unique_id:04d}",
                    'Artist': artist,
                    'Song': song,
                    'Dialect': song_lang,
                    'Lyrics': lyrics
                })
                print(f"Added {unique_id:04d}: {artist} - {song} ({song_lang})")
                unique_id += 1
                if unique_id > 9999:
                    break
        if unique_id > 9999:
            break
    
    df = pd.DataFrame(all_data)
    df.to_csv('songs_lyrics.csv', index=False)

if __name__ == "__main__":
    main()

Processing Arabic (Egyptian)...
Added 0000: Ragheb Alama - نسيني الدنيا (Nassini El Donia) (Egyptian)
Added 0001: Mahmoud El-Lithy - سطلانة (Satalana) (Egyptian)
Added 0002: Elissa - أجمل إحساس (Agmal Ehsas) (Egyptian)
Added 0003: Sabah - يانا يانا (Yana Yana) (Egyptian)
Added 0004: Hassan Shakosh - حبيبتي (Habibty) (Egyptian)
Added 0005: Abdel Halim Hafez - زي الهوا (Zay El Hawa) (Egyptian)
Added 0006: Amr Diab - قمرين (Amarain) (Egyptian)
Added 0007: Oum Kalthoum - دارت الأيام (Daret El Ayam) (Egyptian)
Added 0008: Ragheb Alama - حبيب قلبى (Habib Albi) (Egyptian)
Added 0009: Cairokee - تلك قضية (Telk Qadeya) (Egyptian)
Added 0010: Maher Zain - محمد (ص) [واحشنا] (Muhammad (Pbuh) [Waheshna]) (Egyptian)
Added 0011: Tangled (OST) - أخيرا شفت النور [I See The Light] (Akhiran shuft en-nor) (Egyptian)
Added 0012: Aladdin (OST) - دى دنيا فوق [A Whole New World] (Di dunya fu') (Egyptian)
Added 0013: Fadel Chaker - أحلى رسمة (Ahla Rasma) (Egyptian)
Added 0014: Ruby (Egypt) - ليه بيداري (Leih B

# Data Cleaning

In [19]:
import pandas as pd

# Load your CSV
df = pd.read_csv("songs_lyrics.csv")

# Check duplicates in the Lyrics column
duplicates = df[df.duplicated(subset=["Lyrics"], keep=False)]
print("Found duplicate lyrics:\n")
print(duplicates)

# Remove duplicates (keep the first occurrence only)
df_no_duplicates = df.drop_duplicates(subset=["Lyrics"], keep="first")

# Save to new CSV
df_no_duplicates.to_csv("songs_no_duplicates.csv", index=False)

print("\nDuplicates removed. Saved as songs_no_duplicates.csv")


Found duplicate lyrics:

      ID           Artist                                               Song  \
55    55     Bahaa Soltan                    صاحبي يا صاحبي (Sahby Ya Sahby)   
264  264     Bahaa Soltan  صاحبي يا صاحبي (النسخة الحزينة) (Sahby Ya Sahb...   
523  523      Hoda Haddad                         وديلي سلام (Waddeli Salam)   
524  524   Lena Chamamyan                                 يا ظريف (Ya zarif)   
525  525      Hoda Haddad            اسمي خبيتو بنسمي (Esmi Khabaito bnesmi)   
..   ...              ...                                                ...   
853  853     Yousra Saouf                        بغيتو حلالي (Bghito Halali)   
854  854  Manal (Morocco)                                       دنيا (Denia)   
855  855      Weld Lgriya                            عاش الشعب (3acha cha3b)   
856  856     Yousra Saouf                     حبيبي مختلف (Habiby Mokhtalef)   
857  857           Khaled  وهران لمارساي (أورا ميكس) (Oran Marseille (Ora...   

       Dialect

In [21]:
import pandas as pd
import re

# Load CSV
df = pd.read_csv("songs_no_duplicates.csv")

# Function to remove Arabic diacritics
def remove_diacritics(text):
    return re.sub(r"[\u064B-\u065F]", "", str(text))

# Strip whitespace and remove diacritics from Lyrics
df["Lyrics"] = df["Lyrics"].astype(str).str.strip().apply(remove_diacritics)

# Optional: also strip Song names (without removing diacritics there)
df["Song"] = df["Song"].astype(str).str.strip()

# Save cleaned file
df.to_csv("songs_cleaned.csv", index=False)

print("Cleaning done! Saved as songs_cleaned.csv")


Cleaning done! Saved as songs_cleaned.csv


# Data ....